# Demo 7 — Dal rumore a un piccolo pop

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 44100
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
SR_R = 10_000
rng = np.random.default_rng(2026)
t_pop = np.arange(SR_R//2)/SR_R
burst = 0.2*rng.uniform(-1, 1, len(t_pop))*np.exp(-t_pop/0.008)
f_pop, B_pop = 400., 80.
R_pop = 1-np.pi*B_pop/SR_R  # progetto approssimato di banda stretta
w_pop = 2*np.pi*f_pop/SR_R
G_pop = (1-R_pop)*np.sqrt(1-2*R_pop*np.cos(2*w_pop)+R_pop**2)
pop = al.filter_signal([G_pop], [1., -2*R_pop*np.cos(w_pop), R_pop**2], burst)
al.compare_audio([("Eccitazione: rumore breve", burst), ("Rumore nel risuonatore", pop)], SR_R)
fig = go.Figure()
for nome, valori in (("Eccitazione", burst), ("Uscita", pop)):
    fig.add_trace(go.Scatter(x=t_pop, y=valori, name=nome))
fig.update_xaxes(title_text="Tempo [s]", range=[0, 0.12])
fig.update_yaxes(title_text="Ampiezza")
fig.update_layout(height=320, title="Un'eccitazione breve mette in moto la risonanza")
fig.show()
